In [36]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

True

In [37]:
src_engine = create_engine(
    f"mssql+pyodbc://@{os.getenv('SQLSERVER_HOST')}/{os.getenv('SQLSERVER_DB')}"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&trusted_connection=yes"
    "&TrustServerCertificate=yes"
)
pd.read_sql("SELECT COUNT(*) AS row_count FROM patients", src_engine)

,row_count
0,1163


In [38]:
tgt_engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('PG_USER')}:{os.getenv('PG_PASSWORD')}"
    f"@{os.getenv('PG_HOST')}:{os.getenv('PG_PORT')}/{os.getenv('PG_DB')}"
)
pd.read_sql("SELECT COUNT(*) AS row_count FROM patients", tgt_engine)

,row_count
0,1163


In [6]:
df = pd.read_sql("SELECT * FROM patients", src_engine)
print(df.shape)
df.dtypes

(1163, 25)


Id                         str
BIRTHDATE               object
DEATHDATE               object
SSN                        str
DRIVERS                    str
PASSPORT                   str
PREFIX                     str
FIRST                      str
LAST                       str
SUFFIX                     str
MAIDEN                     str
MARITAL                    str
RACE                       str
ETHNICITY                  str
GENDER                     str
BIRTHPLACE                 str
ADDRESS                    str
CITY                       str
STATE                      str
COUNTY                     str
ZIP                        str
LAT                    float64
LON                    float64
HEALTHCARE_EXPENSES    float64
HEALTHCARE_COVERAGE    float64
dtype: object

In [15]:
df.columns = df.columns.str.lower()

df["healthcare_expenses"] = df["healthcare_expenses"].round(2)
df["healthcare_coverage"] = df["healthcare_coverage"].round(2)

df["zip"].str.len().value_counts()

zip
5.0    618
Name: count, dtype: int64

In [17]:
compare = pd.DataFrame({
    "csv_nulls": csv.isna().sum().values,
    "sql_nulls": df.isna().sum().values
}, index=df.columns)

compare[compare["csv_nulls"] != compare["sql_nulls"]]

,csv_nulls,sql_nulls


In [18]:
df.to_sql("patients", tgt_engine, if_exists="append", index=False)
print("Loaded", len(df), "rows")

Loaded 1163 rows


In [19]:
src_count = pd.read_sql("SELECT COUNT(*) AS n FROM patients", src_engine)["n"][0]
tgt_count = pd.read_sql("SELECT COUNT(*) AS n FROM patients", tgt_engine)["n"][0]
print("Source:", src_count, "| Target:", tgt_count, "| Match:", src_count == tgt_count)

Source: 1163 | Target: 1163 | Match: True


In [20]:
src_engine = create_engine(
    f"mssql+pyodbc://@{os.getenv('SQLSERVER_HOST')}/{os.getenv('SQLSERVER_DB')}"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&trusted_connection=yes"
    "&TrustServerCertificate=yes"
)
pd.read_sql("SELECT COUNT(*) AS row_count FROM encounters", src_engine)

,row_count
0,61459


In [22]:
tgt_engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('PG_USER')}:{os.getenv('PG_PASSWORD')}"
    f"@{os.getenv('PG_HOST')}:{os.getenv('PG_PORT')}/{os.getenv('PG_DB')}"
)
pd.read_sql("SELECT COUNT(*) AS row_count FROM encounters", tgt_engine)

,row_count
0,0


In [23]:
df = pd.read_sql("SELECT * FROM encounters", src_engine)
print(df.shape)
df.dtypes

(61459, 15)


Id                         str
START                      str
STOP                       str
PATIENT                    str
ORGANIZATION               str
PROVIDER                   str
PAYER                      str
ENCOUNTERCLASS             str
CODE                       str
DESCRIPTION                str
BASE_ENCOUNTER_COST    float64
TOTAL_CLAIM_COST       float64
PAYER_COVERAGE         float64
REASONCODE                 str
REASONDESCRIPTION          str
dtype: object

In [25]:
df.columns = df.columns.str.lower()

for col in ["base_encounter_cost", "total_claim_cost", "payer_coverage"]:
    df[col] = df[col].round(2)

for col in ["start", "stop"]:
    df[col] = pd.to_datetime(df[col], utc=True).dt.tz_localize(None)

df.dtypes

id                                str
start                  datetime64[us]
stop                   datetime64[us]
patient                           str
organization                      str
provider                          str
payer                             str
encounterclass                    str
code                              str
description                       str
base_encounter_cost           float64
total_claim_cost              float64
payer_coverage                float64
reasoncode                        str
reasondescription                 str
dtype: object

In [26]:
df.to_sql("encounters", tgt_engine, if_exists="append", index=False)
print("Loaded", len(df), "rows")

Loaded 61459 rows


In [27]:
src_count = pd.read_sql("SELECT COUNT(*) AS n FROM encounters", src_engine)["n"][0]
tgt_count = pd.read_sql("SELECT COUNT(*) AS n FROM encounters", tgt_engine)["n"][0]
print("Source:", src_count, "| Target:", tgt_count, "| Match:", src_count == tgt_count)

Source: 61459 | Target: 61459 | Match: True


In [28]:
def migrate_table(table, date_cols=(), money_cols=(), int_cols=(), chunksize=50000):
    total = 0

    with tgt_engine.begin() as conn:
        for chunk in pd.read_sql(f"SELECT * FROM {table}", src_engine, chunksize=chunksize):
            # TRANSFORM
            chunk.columns = chunk.columns.str.lower()
            for col in date_cols:
                chunk[col] = pd.to_datetime(chunk[col], utc=True).dt.tz_localize(None)
            for col in money_cols:
                chunk[col] = chunk[col].round(2)
            for col in int_cols:
                chunk[col] = pd.to_numeric(chunk[col]).astype("Int64")

            # LOAD
            chunk.to_sql(table, conn, if_exists="append", index=False)
            total += len(chunk)
            print(f"{table}: {total} rows loaded")

    # VALIDATE
    src = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", src_engine)["n"][0]
    tgt = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", tgt_engine)["n"][0]
    print(f"{table} -> Source: {src} | Target: {tgt} | Match: {src == tgt}")

In [29]:
migrate_table("conditions", date_cols=("start", "stop"))

conditions: 38094 rows loaded
conditions -> Source: 38094 | Target: 38094 | Match: True


In [30]:
migrate_table("medications",
              date_cols=("start", "stop"),
              money_cols=("base_cost", "payer_coverage", "totalcost"),
              int_cols=("dispenses",))

medications: 50000 rows loaded
medications: 56430 rows loaded
medications -> Source: 56430 | Target: 56430 | Match: True


In [31]:
migrate_table("observations", date_cols=("date",))

observations: 50000 rows loaded
observations: 100000 rows loaded
observations: 150000 rows loaded
observations: 200000 rows loaded
observations: 250000 rows loaded
observations: 300000 rows loaded
observations: 350000 rows loaded
observations: 400000 rows loaded
observations: 450000 rows loaded
observations: 500000 rows loaded
observations: 531144 rows loaded
observations -> Source: 531144 | Target: 531144 | Match: True


In [34]:
pd.options.display.float_format = "{:,.2f}".format


In [39]:
checks = {
    "patients":     {"date": "birthdate", "money": "healthcare_expenses", "key": "id"},
    "encounters":   {"date": "start",     "money": "total_claim_cost",    "key": "patient"},
    "conditions":   {"date": "start",     "money": None,                  "key": "patient"},
    "medications":  {"date": "start",     "money": "totalcost",           "key": "patient"},
    "observations": {"date": "date",      "money": None,                  "key": "patient"},
}

def profile(engine, table, c):
    money = f"SUM({c['money']})" if c["money"] else "NULL"
    q = f"""
        SELECT COUNT(*) AS row_count,
               COUNT(DISTINCT {c['key']}) AS distinct_patients,
               MIN({c['date']}) AS min_date,
               MAX({c['date']}) AS max_date,
               {money} AS money_total
        FROM {table}
    """
    return pd.read_sql(q, engine).iloc[0]

def to_dt(v):
    return pd.to_datetime(v, utc=True).tz_localize(None)

rows = []
for table, c in checks.items():
    s = profile(src_engine, table, c)
    t = profile(tgt_engine, table, c)
    rows.append({
        "table": table,
        "rows_match": s.row_count == t.row_count,
        "patients_match": s.distinct_patients == t.distinct_patients,
        "dates_match": to_dt(s.min_date) == to_dt(t.min_date) and to_dt(s.max_date) == to_dt(t.max_date),
        "money_src": round(float(s.money_total), 2) if c["money"] else None,
        "money_tgt": round(float(t.money_total), 2) if c["money"] else None,
    })

report = pd.DataFrame(rows)
report["money_diff"] = report["money_tgt"] - report["money_src"]
report.to_csv("validation_report.csv", index=False)
report

,table,rows_match,patients_match,dates_match,money_src,money_tgt,money_diff
0,patients,True,True,True,"1,461,016,960.62","1,461,016,960.52",-0.10
1,encounters,True,True,True,"255,033,828.03","255,033,828.04",0.01
2,conditions,True,True,True,NaN,NaN,NaN
3,medications,True,True,True,"73,907,812.43","73,907,812.43",0.00
4,observations,True,True,True,NaN,NaN,NaN
